# Mask vs No-Mask Detection — YOLOv8 Fine-Tuning

**What this is:** A face-mask detector built by fine-tuning a pretrained **YOLOv8n/YOLOv8s** model
(Ultralytics) on a public [Roboflow Universe dataset](https://universe.roboflow.com/roboflow-universe-projects/mask-wearing-iskms).
This is **not** trained from scratch — the base weights are pretrained on COCO, and only fine-tuned here
on the mask dataset. Credit to the dataset creators (`roboflow-universe-projects / mask-wearing-iskms`).

**Pipeline covered in this notebook:**
1. Secure setup (no hardcoded keys) + dataset download & inspection
2. Train YOLOv8n, validate on val/test splits
3. Visual results: confusion matrix + PR curve
4. Model comparison: YOLOv8n vs YOLOv8s (accuracy/speed tradeoff)
5. Confidence-threshold experiment
6. Inference speed benchmark (CPU vs GPU)
7. Single-image test + ONNX export for deployment
8. Group-photo detection + counting, with a before/after visualization

**Known limitations (read before you trust this in production):**
- Dataset is COVID-era and relatively small — may not generalize well to side-profile faces,
  low light, or unusual mask types (e.g. patterned masks, face shields).
- Class balance between "Mask" and "NO-Mask" was not explicitly checked/corrected for.
- Not tested on video streams, only static images.


## 1. Setup, secure API key, and dataset download

In [ ]:
# ============================================================
# CELL: Install libraries and check GPU
# ============================================================
!pip install -q roboflow ultralytics pyyaml gradio

import os
import yaml
import torch
import time

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — go to Runtime > Change runtime type > GPU")


In [ ]:
# ============================================================
# CELL: SECURE API key loading — never hardcode keys in a notebook
# ============================================================
# Option A (Google Colab): store the key using the Secrets manager (key icon in
#   the left sidebar), name it ROBOFLOW_API_KEY, then this line picks it up.
# Option B (local / other envs): set an environment variable before launching
#   Jupyter, e.g.  export ROBOFLOW_API_KEY="your_key_here"
#
# IMPORTANT: if you ever pasted a real key into a notebook before, treat it as
# compromised — regenerate/revoke it in your Roboflow account settings.

API_KEY = None

try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    pass

if not API_KEY:
    API_KEY = os.environ.get("ROBOFLOW_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "No Roboflow API key found. Set it via Colab Secrets (ROBOFLOW_API_KEY) "
        "or as an environment variable before running this cell."
    )

print("[OK] API key loaded securely (value not printed).")


In [ ]:
# ============================================================
# CELL: Download dataset, clean up, inspect classes & split sizes
# ============================================================
from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("roboflow-universe-projects").project("mask-wearing-iskms")
version = project.version(6)
dataset = version.download("yolov8")

print("\n[OK] Dataset downloaded to:", dataset.location)

# Clean up the leftover zip file to save disk space
import glob
for zip_path in glob.glob("/content/*.zip"):
    os.remove(zip_path)
    print("[SPACE] Removed:", zip_path)

# Inspect classes and split sizes directly from data.yaml (never guess this)
yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

num_classes = data_cfg.get("nc")
class_names = data_cfg.get("names")

print("\n================ DATASET SUMMARY ================")
print("Number of classes:", num_classes)
print("Class names      :", class_names)

split_counts = {}
for split in ["train", "valid", "test"]:
    split_path = data_cfg.get(split)
    if split_path:
        full_path = os.path.join(dataset.location, split_path)
        if os.path.isdir(full_path):
            n_images = len([f for f in os.listdir(full_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            split_counts[split] = n_images
            print(f"{split:<6} images  : {n_images}")
print("===================================================")


In [ ]:
# ============================================================
# CELL: Check class balance (Mask vs NO-Mask) in the training set
# ============================================================
# An imbalanced dataset (e.g. way more "Mask" than "NO-Mask" examples) will
# bias the model toward the majority class. Good to know and disclose.
from collections import Counter

train_labels_dir = os.path.join(dataset.location, data_cfg.get("train")).replace("images", "labels")
label_counts = Counter()

if os.path.isdir(train_labels_dir):
    for fname in os.listdir(train_labels_dir):
        if fname.endswith(".txt"):
            with open(os.path.join(train_labels_dir, fname)) as f:
                for line in f:
                    cls_id = int(line.split()[0])
                    label_counts[class_names[cls_id]] += 1

print("================ CLASS BALANCE (train) ================")
for cls, count in label_counts.items():
    print(f"{cls:<10}: {count}")
print("=========================================================")


## 2. Train YOLOv8n and evaluate on val/test

In [ ]:
# ============================================================
# CELL: Train YOLOv8n, then validate and test it
# ============================================================
from ultralytics import YOLO

model_n = YOLO("yolov8n.pt")

results_n = model_n.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=-1,          # auto-sized to your GPU memory
    patience=15,       # early stopping if it stops improving
    cache="disk",
    workers=2,
    amp=True,
    rect=True,
    plots=True,        # saves PR-curve / confusion matrix for review
    name="mask_detector_yolov8n",
    exist_ok=True,
    save_period=-1,
)

print("\n[OK] YOLOv8n training complete. Best weights at:", model_n.trainer.best)
torch.cuda.empty_cache()

val_metrics_n = model_n.val(data=os.path.join(dataset.location, "data.yaml"), split="val")
print("\n================ YOLOv8n VALIDATION METRICS ================")
print(f"mAP50-95 : {val_metrics_n.box.map:.4f}")
print(f"mAP50    : {val_metrics_n.box.map50:.4f}")
print(f"Precision: {val_metrics_n.box.mp:.4f}")
print(f"Recall   : {val_metrics_n.box.mr:.4f}")

test_metrics_n = model_n.val(data=os.path.join(dataset.location, "data.yaml"), split="test")
print("\n================ YOLOv8n TEST METRICS ================")
print(f"mAP50-95 : {test_metrics_n.box.map:.4f}")
print(f"mAP50    : {test_metrics_n.box.map50:.4f}")
print(f"Precision: {test_metrics_n.box.mp:.4f}")
print(f"Recall   : {test_metrics_n.box.mr:.4f}")


## 3. Visual results: confusion matrix + PR curve

Ultralytics automatically saves these plots during training/validation. We display them inline so the LinkedIn post has actual evidence, not just printed numbers.

In [ ]:
# ============================================================
# CELL: Show confusion matrix and PR curve generated during training
# ============================================================
from IPython.display import Image as IPImage, display

run_dir = model_n.trainer.save_dir  # e.g. runs/detect/mask_detector_yolov8n

confusion_matrix_path = os.path.join(run_dir, "confusion_matrix.png")
pr_curve_path = os.path.join(run_dir, "BoxPR_curve.png")

for path, title in [(confusion_matrix_path, "Confusion Matrix"),
                     (pr_curve_path, "Precision-Recall Curve")]:
    if os.path.exists(path):
        print(f"--- {title} ---")
        display(IPImage(filename=path, width=600))
    else:
        print(f"[WARN] {title} not found at {path}. Check runs/detect/ for the exact folder name.")


## 4. Model comparison: YOLOv8n vs YOLOv8s

A slightly larger backbone (YOLOv8s) usually trades some speed for better accuracy. Comparing both shows you understand the tradeoff rather than picking a model size at random.

In [ ]:
# ============================================================
# CELL: Train YOLOv8s for comparison against YOLOv8n
# ============================================================
model_s = YOLO("yolov8s.pt")

results_s = model_s.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=-1,
    patience=15,
    cache="disk",
    workers=2,
    amp=True,
    rect=True,
    plots=True,
    name="mask_detector_yolov8s",
    exist_ok=True,
    save_period=-1,
)

print("\n[OK] YOLOv8s training complete. Best weights at:", model_s.trainer.best)
torch.cuda.empty_cache()

test_metrics_s = model_s.val(data=os.path.join(dataset.location, "data.yaml"), split="test")
print("\n================ YOLOv8s TEST METRICS ================")
print(f"mAP50-95 : {test_metrics_s.box.map:.4f}")
print(f"mAP50    : {test_metrics_s.box.map50:.4f}")
print(f"Precision: {test_metrics_s.box.mp:.4f}")
print(f"Recall   : {test_metrics_s.box.mr:.4f}")


In [ ]:
# ============================================================
# CELL: Side-by-side comparison table (YOLOv8n vs YOLOv8s)
# ============================================================
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["YOLOv8n", "YOLOv8s"],
    "mAP50-95": [test_metrics_n.box.map, test_metrics_s.box.map],
    "mAP50": [test_metrics_n.box.map50, test_metrics_s.box.map50],
    "Precision": [test_metrics_n.box.mp, test_metrics_s.box.mp],
    "Recall": [test_metrics_n.box.mr, test_metrics_s.box.mr],
    "Params (approx, millions)": [3.2, 11.2],
})

print("================ MODEL COMPARISON ================")
print(comparison.to_string(index=False))
print("=====================================================")
print("\nTakeaway: pick YOLOv8s if the accuracy gain matters more than speed/size; ")
print("pick YOLOv8n if you need this running on a lightweight/edge device.")

# Choose the model to carry forward for the rest of the notebook
model = model_s if test_metrics_s.box.map50 > test_metrics_n.box.map50 else model_n
print(f"\n[INFO] Using {'YOLOv8s' if model is model_s else 'YOLOv8n'} for the remaining steps.")


## 5. Confidence-threshold experiment

The confidence threshold controls the tradeoff between catching more faces (lower threshold, more false positives) and being stricter (higher threshold, may miss faces). Testing a couple of values makes this tradeoff visible instead of just picking `0.25` blindly.

In [ ]:
# ============================================================
# CELL: Run test-set validation at a few confidence thresholds
# ============================================================
thresholds = [0.10, 0.25, 0.50, 0.70]
threshold_results = []

for conf in thresholds:
    m = model.val(data=os.path.join(dataset.location, "data.yaml"), split="test", conf=conf)
    threshold_results.append({
        "confidence_threshold": conf,
        "precision": m.box.mp,
        "recall": m.box.mr,
        "mAP50": m.box.map50,
    })

conf_df = pd.DataFrame(threshold_results)
print("================ CONFIDENCE THRESHOLD SWEEP ================")
print(conf_df.to_string(index=False))
print("===============================================================")
print("\nTakeaway: lower thresholds boost recall (catch more faces) at the cost of ")
print("precision (more false positives), and vice versa for higher thresholds.")


## 6. Inference speed benchmark (CPU vs GPU)

For a real-time detector, speed (FPS) often matters as much as accuracy. This gives an honest, measured number instead of just claiming "it's fast".

In [ ]:
# ============================================================
# CELL: Benchmark inference speed on GPU (if available) and CPU
# ============================================================
import numpy as np

dummy_image = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
n_runs = 30

def benchmark(model, device, n_runs=30):
    model.to(device)
    # warm-up
    for _ in range(5):
        _ = model(dummy_image, imgsz=640, verbose=False, device=device)
    start = time.time()
    for _ in range(n_runs):
        _ = model(dummy_image, imgsz=640, verbose=False, device=device)
    elapsed = time.time() - start
    fps = n_runs / elapsed
    return fps

print("================ INFERENCE SPEED BENCHMARK ================")
if torch.cuda.is_available():
    fps_gpu = benchmark(model, device=0, n_runs=n_runs)
    print(f"GPU: {fps_gpu:.1f} FPS  ({1000/fps_gpu:.1f} ms/frame)")
else:
    print("GPU not available in this runtime — skipping GPU benchmark.")

fps_cpu = benchmark(model, device="cpu", n_runs=n_runs)
print(f"CPU: {fps_cpu:.1f} FPS  ({1000/fps_cpu:.1f} ms/frame)")
print("===============================================================")


## 7. Single-image test + export for deployment

In [ ]:
# ============================================================
# CELL: Test on your own photo, then export the model
# ============================================================
from google.colab import files

print("Upload a photo of a face (masked or unmasked) to test the model:")
uploaded = files.upload()
test_image_path = list(uploaded.keys())[0]

results = model(test_image_path, imgsz=640)
results[0].show()                                   # displays boxes inline
results[0].save(filename="prediction_output.jpg")   # also saves to disk
print("[OK] Saved annotated result as prediction_output.jpg")

# --- Export a lightweight deployable copy ---
onnx_path = model.export(format="onnx", simplify=True, half=True)
print("[OK] ONNX model exported to:", onnx_path)

# --- Download the trained weights to your computer ---
files.download(str(model.trainer.best))   # best.pt
files.download(onnx_path)                 # onnx version


## 8. Group-photo detection + counting, with a before/after visual

This is the part worth showing on LinkedIn: a raw group photo next to the annotated output with a Mask / NO-Mask count overlay.

In [ ]:
# ============================================================
# CELL: Detect + count Mask vs NO-Mask in a group photo, with before/after view
# ============================================================
from PIL import Image, ImageDraw, ImageFont
from collections import Counter
import matplotlib.pyplot as plt

print("Upload a group photo (e.g. 10 people) to test the model:")
uploaded = files.upload()
group_image_path = list(uploaded.keys())[0]

# Run detection — finds every face in the image, one box per person
results = model(group_image_path, imgsz=640, conf=0.25)  # conf=0.25: catches more
                                                            # faces, at the cost of
                                                            # a few more false positives

boxes = results[0].boxes
class_ids = boxes.cls.tolist()
class_names_map = results[0].names             # e.g. {0: 'Mask', 1: 'NO-Mask'}
detected_labels = [class_names_map[int(c)] for c in class_ids]

counts = Counter(detected_labels)
total_people = len(detected_labels)

print("\n================ DETECTION SUMMARY ================")
print(f"Total people detected : {total_people}")
for label, count in counts.items():
    print(f"{label:<10}: {count}")
print("=====================================================")

# --- Draw boxes + labels on a copy of the image ---
original_img = Image.open(group_image_path).convert("RGB")
annotated_img = original_img.copy()
draw = ImageDraw.Draw(annotated_img)

colors = {"Mask": "green", "NO-Mask": "red"}
for box, label in zip(boxes.xyxy.tolist(), detected_labels):
    x1, y1, x2, y2 = box
    color = colors.get(label, "yellow")
    draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
    draw.text((x1, max(y1 - 15, 0)), label, fill=color)

summary_text = "  |  ".join(f"{label}: {count}" for label, count in counts.items())
draw.text((10, 10), f"Total: {total_people}   {summary_text}", fill="white")

annotated_img.save("group_photo_annotated.jpg")
print("[OK] Saved annotated result as group_photo_annotated.jpg")

# --- Before / after side-by-side plot, good for a LinkedIn screenshot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(original_img)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(annotated_img)
axes[1].set_title(f"Detected — {summary_text}")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("before_after_comparison.jpg", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Saved before/after comparison as before_after_comparison.jpg")

files.download("before_after_comparison.jpg")


## 9. (Optional) Quick interactive demo with Gradio

Wraps the exported model in a minimal web UI so people can actually try it, not just look at a static image. Skip this cell if you only need the notebook results.

In [ ]:
# ============================================================
# CELL: Minimal Gradio demo (optional — useful if hosting on HF Spaces later)
# ============================================================
import gradio as gr

def predict(image):
    results = model.predict(image, imgsz=640, conf=0.25, verbose=False)
    annotated = results[0].plot()  # numpy array with boxes drawn
    return annotated

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Upload a photo"),
    outputs=gr.Image(type="numpy", label="Detected Mask / NO-Mask"),
    title="Mask vs NO-Mask Detector (YOLOv8, fine-tuned)",
    description="Fine-tuned YOLOv8 on a public Roboflow dataset. Upload a face or group photo.",
)

demo.launch(share=True)  # share=True gives a public link if running in Colab


## Summary & honest reflection

- **Base model:** Pretrained YOLOv8n/YOLOv8s (COCO weights), fine-tuned — not trained from scratch.
- **Dataset:** Public Roboflow Universe dataset (`roboflow-universe-projects/mask-wearing-iskms`), not self-collected.
- **What worked well:** Detection and counting on group photos, ONNX export for lightweight deployment.
- **What I'd do differently next time:**
  - Collect / augment more "NO-Mask" examples if the class balance check above showed skew.
  - Test on video, not just static images, since real-world use is often a live camera feed.
  - Try a larger validation set of diverse lighting/angles to get a more reliable mAP estimate.
- **Use case:** Could be adapted for lightweight compliance monitoring (e.g. workplace entry checks),
  with the understanding that it needs more diverse data before any real deployment.
